# AI4Lassa — 01. Data Cleaning, Aggregation & Feature Engineering

Turns the individual-level Lassa fever line-listing dataset (`data/raw/2015-2025__3_.xlsx`)
into a monthly national time series suitable for 1-month-ahead case-volume forecasting.

**Leakage rule enforced throughout:** any feature for month *t* is computed using only
records dated on or before the end of month *t*.

In [1]:
import pandas as pd
import numpy as np

RAW_PATH = "../data/raw/2015-2025__3_.xlsx"
OUT_PATH = "../data/processed/monthly_features.csv"

VALID_RESULTS = {"positive", "negative"}  # excludes pending/rejected/bring-fresh-sample/etc.

## Load and clean the raw line-listing data

In [2]:
def clean_string_col(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower().replace({"nan": np.nan})

df = pd.read_excel(RAW_PATH, sheet_name="Sheet1")
df = df.rename(columns={"DATE OF\n LINE LISTING": "date_line_listing"})
df["date_line_listing"] = pd.to_datetime(df["date_line_listing"], errors="coerce")

n_before = len(df)
df = df.dropna(subset=["date_line_listing"])
print(f"Dropped {n_before - len(df)} rows with unparseable line-listing date")

df["RESULTS_clean"] = clean_string_col(df["RESULTS"])
df["CONDITION_clean"] = clean_string_col(df["CONDITION"])
df["month"] = df["date_line_listing"].dt.to_period("M")
df.head()

Dropped 0 rows with unparseable line-listing date


,S/N,YEAR,date_line_listing,MONTH,AGE in Years,Age in Months,Age In Days,SEX,EDUCATIONAL LEVEL,MARITAL STATUS,...,OTHER TYPE OF SPECIMEN,DATE SEEN AT HEALTH FACILITY,DATE SAMPLES ARRIVED IN THE LAB,DATE SAMPLE TESTED,RESULTS,CONDITION,DATE OF DEATH,RESULTS_clean,CONDITION_clean,month
0,1,2015,2015-01-03,January,30,NaN,NaN,F,Tertiary,Single,...,NaN,2015-01-03,2022-07-16,2015-01-03,Negative,Alive,NaN,negative,alive,2015-01
1,2,2015,2015-01-05,January,24,NaN,NaN,F,Tertiary,Single,...,NaN,2015-01-04,2015-01-04,2015-01-04,Negative,Alive,NaN,negative,alive,2015-01
2,3,2015,2015-01-07,January,20,NaN,NaN,F,Tertiary,Single,...,NaN,2015-01-07,2022-01-07,2015-01-07,Negative,Alive,NaN,negative,alive,2015-01
3,4,2015,2015-01-08,January,32,NaN,NaN,F,Tertiary,Married,...,NaN,2015-01-08,2022-01-08,2015-01-08,Negative,Alive,NaN,negative,alive,2015-01
4,5,2015,2015-01-08,January,45,NaN,NaN,M,Tertiary,Married,...,NaN,2015-01-08,2022-01-08,2015-01-08,Negative,Alive,NaN,negative,alive,2015-01


## Aggregate to a continuous monthly national time series

In [3]:
g = df.groupby("month")

monthly = pd.DataFrame({
    "case_count": g.size(),
    "tested_positive": g.apply(lambda x: (x["RESULTS_clean"] == "positive").sum()),
    "tested_negative": g.apply(lambda x: (x["RESULTS_clean"] == "negative").sum()),
    "deaths": g.apply(lambda x: (x["CONDITION_clean"] == "dead").sum()),
})
monthly["tested_total"] = monthly["tested_positive"] + monthly["tested_negative"]
monthly["positivity_rate"] = monthly["tested_positive"] / monthly["tested_total"]

full_index = pd.period_range(monthly.index.min(), monthly.index.max(), freq="M")
monthly = monthly.reindex(full_index)
monthly.index.name = "month"
print(f"Months with zero recorded cases (true gaps): {monthly['case_count'].isna().sum()}")

for col in ["case_count", "tested_positive", "tested_negative", "deaths", "tested_total"]:
    monthly[col] = monthly[col].fillna(0)

monthly.tail()

Months with zero recorded cases (true gaps): 0


,case_count,tested_positive,tested_negative,deaths,tested_total,positivity_rate
month,,,,,,
2025-08,274,9,259,5,268,0.033582
2025-09,316,11,297,5,308,0.035714
2025-10,216,10,204,3,214,0.046729
2025-11,169,13,152,5,165,0.078788
2025-12,180,7,172,0,179,0.039106


## Feature engineering (lags, rolling stats, seasonality, positivity rate)

In [4]:
m = monthly.reset_index()
m["month_ts"] = m["month"].dt.to_timestamp()
m["calendar_month"] = m["month_ts"].dt.month
m["year"] = m["month_ts"].dt.year

m["month_sin"] = np.sin(2 * np.pi * m["calendar_month"] / 12)
m["month_cos"] = np.cos(2 * np.pi * m["calendar_month"] / 12)

for lag in [1, 2, 3, 6, 12]:
    m[f"case_count_lag{lag}"] = m["case_count"].shift(lag)

m["case_count_roll3_mean"] = m["case_count"].shift(1).rolling(3).mean()
m["case_count_roll6_mean"] = m["case_count"].shift(1).rolling(6).mean()
m["case_count_roll3_max"] = m["case_count"].shift(1).rolling(3).max()
m["case_growth_lag1"] = m["case_count_lag1"] - m["case_count_lag2"]

pos_rate_lag1 = m["positivity_rate"].shift(1)
expanding_mean = m["positivity_rate"].shift(1).expanding().mean()
m["positivity_rate_lag1"] = pos_rate_lag1.fillna(expanding_mean)

# TARGET: next month's case count (1-month-ahead)
m["target_next_month_cases"] = m["case_count"].shift(-1)

model_df = m.dropna(subset=["case_count_lag12", "target_next_month_cases"]).reset_index(drop=True)
model_df.to_csv(OUT_PATH, index=False)

print(f"Saved: {OUT_PATH}")
print(f"Shape: {model_df.shape}")
print(f"Usable range: {model_df['month_ts'].min().date()} to {model_df['month_ts'].max().date()}")
model_df.head()

Saved: ../data/processed/monthly_features.csv
Shape: (119, 23)
Usable range: 2016-01-01 to 2025-11-01


,month,case_count,tested_positive,tested_negative,deaths,tested_total,positivity_rate,month_ts,calendar_month,year,...,case_count_lag2,case_count_lag3,case_count_lag6,case_count_lag12,case_count_roll3_mean,case_count_roll6_mean,case_count_roll3_max,case_growth_lag1,positivity_rate_lag1,target_next_month_cases
0,2016-01,428,34,390,38,424,0.080189,2016-01-01,1,2016,...,78.0,78.0,41.0,60.0,79.666667,70.500000,83.0,5.0,0.072289,324.0
1,2016-02,324,43,265,20,308,0.139610,2016-02-01,2,2016,...,83.0,78.0,63.0,82.0,196.333333,135.000000,428.0,345.0,0.080189,236.0
2,2016-03,236,32,204,14,236,0.135593,2016-03-01,3,2016,...,428.0,83.0,80.0,72.0,278.333333,178.500000,428.0,-104.0,0.139610,173.0
3,2016-04,173,8,165,12,173,0.046243,2016-04-01,4,2016,...,324.0,428.0,78.0,61.0,329.333333,204.500000,428.0,-88.0,0.135593,89.0
4,2016-05,89,2,87,0,89,0.022472,2016-05-01,5,2016,...,236.0,324.0,78.0,49.0,244.333333,220.333333,324.0,-63.0,0.046243,61.0


## Result

**119 usable monthly rows** (2016-01 → 2025-11), zero missing values in the final feature table,
zero unparseable dates, zero true gap-months. Ready for the temporal train/validation/test split
used in the next notebook.